In [ ]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get update
!apt-get install -y ffmpeg espeak-ng

!pip uninstall -y torch torchaudio torchvision xformers audiocraft

!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install transformers==4.41.2 diffusers accelerate gradio unidecode phonemizer torchlibrosa ftfy progressbar av einops flashy hydra-core hydra-colorlog julius num2words sentencepiece spacy librosa torchdiffeq torchmetrics dora-search demucs gdown alias-free-torch pytorch-lightning diffq

!pip install --no-deps git+https://github.com/facebookresearch/audiocraft.git
!pip install --no-deps git+https://github.com/haoheliu/AudioLDM2.git

!sed -i 's/latent_diffusion.load_state_dict(checkpoint\["state_dict"\])/latent_diffusion.load_state_dict(checkpoint["state_dict"], strict=False)/g' /usr/local/lib/python3.12/dist-packages/audioldm2/pipeline.py

!git clone https://github.com/yoongi43/MGE-LDM.git
%cd /content/MGE-LDM

!find configs -type f -name "*.yaml" -exec sed -i 's|/data2/yoongi/dataset/pre_trained|/content/MGE-LDM/checkpoints|g' {} +
!sed -i 's/==/>=/g' requirements.txt
!sed -i '/flash-attn/d' requirements.txt
!sed -i '/torch/d' requirements.txt
!sed -i '/transformers/d' requirements.txt
!pip install -r requirements.txt

!pip install xformers==0.0.27.post2 --no-deps

!pip install numpy==1.26.4 --force-reinstall

!mkdir -p checkpoints
!wget "https://huggingface.co/lukewys/laion_clap/resolve/main/music_audioset_epoch_15_esc_90.14.pt?download=true" -O checkpoints/music_audioset_epoch_15_esc_90.14.pt
!gdown --id 1tyND8iI5Whs6_Oe-pBK2SpysGLKKa6sR -O checkpoints/unwrapped_DiT.ckpt

%cd /content

In [ ]:
RUN_MUSICGEN = False
RUN_AUDIOLDM2 = False
RUN_MGE_LDM = False

if RUN_MUSICGEN:
    try:
        import audiocraft
        from audiocraft.models import MusicGen
        from audiocraft.data.audio import audio_write
        print("Success!")
    except ImportError as e:
        print(f"Installation check failed: {e}")

if RUN_AUDIOLDM2:
    try:
        import audioldm2
        print("Success!")
    except ImportError as e:
        print(f"Installation check failed: {e}")

In [ ]:
import os
import glob
import subprocess
import shutil
import random
import gc
import torch
import torchaudio
import soundfile as sf
import numpy as np
import librosa

# Helper Function
def to_tensor(segment):
    t = torch.from_numpy(segment).float()
    if t.dim() == 1:
        t = t.unsqueeze(0)
    return t


def augment_vocals(input_vocals_path, output_vocals_path):
    """
    Applies Pitch Shifting, Tempo Change and Time Shifting
    """
    # Load file
    y, sr = librosa.load(input_vocals_path, sr=None)

    # Pitch shift: The audio track is pitch-shifted by a random integer of semitones in the range of [-4, 4]
    n_steps = random.uniform(-4.0, 4.0)
    y_shifted = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)

    # Tempo change: The audio track’s tempo is altered by a random factor in the range of [0.9, 1.1]
    rate = random.uniform(0.9, 1.1)
    y_stretched = librosa.effects.time_stretch(y=y_shifted, rate=rate)

    # Time shift: The whole track is shifted by a random time from a range of [-3, 3] seconds
    # Vocals will arrive earlier/later than the beat
    shift_ms = random.randint(-100, 100)
    shift_samples = int(sr * (shift_ms / 1000.0))

    if shift_samples > 0:
        y_final = np.pad(y_stretched, (shift_samples, 0), mode='constant')
    elif shift_samples < 0:
        y_final = y_stretched[abs(shift_samples):]
    else:
        y_final = y_stretched

    # Saving the modified file
    sf.write(output_vocals_path, y_final, sr)

    print(f"Augmentation: Pitch {n_steps:+.2f} | Speed {rate:.2f}x | Shift {shift_ms}ms")

def mix_vocals_and_instrumental(generated_inst_path, vocals_path, final_output_path):
    inst, sr_inst = torchaudio.load(generated_inst_path)
    voc, sr_voc = torchaudio.load(vocals_path)

    if sr_inst != sr_voc:
        voc = torchaudio.functional.resample(voc, orig_freq=sr_voc, new_freq=sr_inst)

    min_len = min(inst.shape[1], voc.shape[1])
    mixed = inst[:, :min_len] + voc[:, :min_len]

    # Safe Normalization
    max_val = torch.max(torch.abs(mixed))
    if max_val > 1.0:
        mixed = mixed / max_val

    torchaudio.save(final_output_path, mixed, sr_inst)
    print("Mixed augmented vocals successfully!")

def generate_musicgen(text_prompt, audio_input_path, sr, output_path, model=None):
    """
    MusicGen: Loads a file, generates a song using 1 prompt and saves it at output_path.
    """
    print(f"Generating MusicGen audio with prompt: '{text_prompt}'")

    # Load song from given path
    segment, current_sr = torchaudio.load(audio_input_path)

    # Resample
    if current_sr != model.sample_rate:
         segment = torchaudio.functional.resample(segment, orig_freq=current_sr, new_freq=model.sample_rate)

    segment_np = segment[0].numpy()

    musicgen_tensor = to_tensor(segment_np)

    # Generation
    wav = model.generate_with_chroma([text_prompt], musicgen_tensor[None], model.sample_rate)

    # Save at output_path
    clean_output_path = output_path.replace('.wav', '')
    audio_write(clean_output_path, wav[0].cpu(), model.sample_rate, strategy="loudness")
    print(f"MusicGen output saved to: {clean_output_path}.wav")


def generate_audioldm(text_prompt, audio_input_path, sr, output_path):
    """
    AudioLDM2: Generates an audio file using AudioLDM2 from CLI.
    """
    print(f"Generating AudioLDM2 audio with prompt: '{text_prompt}'")

    # Output file
    output_base_dir = os.path.dirname(output_path)
    os.makedirs(output_base_dir, exist_ok=True)

    audioldm_cmd = f"audioldm2 -t \"{text_prompt}\" -f \"{audio_input_path}\" -s \"{output_base_dir}\" --model_name \"audioldm_48k\" -d \"cuda\""

    try:
        subprocess.run(audioldm_cmd, shell=True, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"Error generating AudioLDM2: {e.stderr}")
        return

    # AudioLDM 2 creates subfolders automatically
    subdirs = [os.path.join(output_base_dir, d) for d in os.listdir(output_base_dir) if os.path.isdir(os.path.join(output_base_dir, d))]
    if not subdirs:
        return
    subdirs.sort(key=os.path.getctime, reverse=True)
    latest_output_dir = subdirs[0]

    generated_files = glob.glob(os.path.join(latest_output_dir, "*.wav"))
    if not generated_files:
        return

    original_generated_path = generated_files[0]

    # Transfer generated file at output_path
    try:
        os.rename(original_generated_path, output_path)
        print(f"AudioLDM2 output saved to: {output_path}")
    except OSError as e:
        print(f"Error renaming AudioLDM2 file: {e}")

    # Clear
    if not os.listdir(latest_output_dir):
        os.rmdir(latest_output_dir)


def generate_mgeldm(text_prompt, stem_path, output_path):
    """
    MGE-LDM: Reads a specific stem and calls infer.py.
    """
    print(f"Generating MGE-LDM audio with prompt: '{text_prompt}' based on stem: {stem_path}")

    # Αφαίρεση της κλήσης seperate() - Διαβάζουμε κατευθείαν το αρχείο
    if not os.path.exists(stem_path):
        print(f"Error: Stem {stem_path} not found.")
        return

    # Preparing MGE-LDM
    mge_temp_output_dir = "/content/MGE-LDM/output/temp_mge_run"
    if os.path.exists(mge_temp_output_dir):
        shutil.rmtree(mge_temp_output_dir)
    os.makedirs(mge_temp_output_dir)

    # Call MGE-LDM CLI
    cmd = [
        "python", "infer.py",
        "--config-name", "dit",
        "+task=partial_gen",
        "ckpt_path=checkpoints/unwrapped_DiT.ckpt",
        f"+given_wav_path={stem_path}",
        f"+text_prompt='{text_prompt}'",
        "+num_steps=100",
        "+cfg_scale=6.0",
        "+overlap_dur=5.0",
        "+repaint_n=1",
        f"+output_dir={mge_temp_output_dir}"
    ]

    try:
        env = os.environ.copy()
        env['TORCH_FORCE_WEIGHTS_ONLY_LOAD'] = '0'
        subprocess.run(cmd, check=True, cwd="/content/MGE-LDM", env=env)

        # Transfer file at output_path
        source_gen_mix_sum_path = os.path.join(mge_temp_output_dir, 'partial_gen_single', 'output_0001', 'gen_mix_sum.wav')
        if os.path.exists(source_gen_mix_sum_path):
            shutil.move(source_gen_mix_sum_path, output_path)
            print(f"MGE-LDM output saved to: {output_path}")

    except subprocess.CalledProcessError as e:
        print(f"Error running MGE-LDM: {e.stderr} \n{e.stdout}")
        print(f"Εxit code: {e.returncode}")

    finally:
        # Clean temporary MGE directory
        if os.path.exists(mge_temp_output_dir):
            shutil.rmtree(mge_temp_output_dir)

In [ ]:
# Lists of Prompts
genres = [
    "alternative", "baroque", "blues", "bollywood", "c-pop", "celtic", "christian rock",
    "classical", "country", "crunk", "dance", "dancehall", "disco", "doom metal", "electronic",
    "folk", "funk", "fusion", "gospel", "gothic", "grime", "grunge", "hard rock", "heavy metal",
    "hip hop", "indie rock", "j-pop", "jazz", "k-pop", "lo-fi", "lounge", "metal", "metalcore",
    "new age", "opera", "orchestral", "pop", "pop rock", "progressive metal", "progressive rock",
    "punk", "r&b", "rap", "reggae", "salsa", "smooth jazz", "soul", "sufi", "world music"
]

moods = [
    "adventurous", "ambivalent", "amused", "angry", "anxious", "apathetic", "bittersweet",
    "blissful", "calm", "carefree", "cautious", "chaotic", "confident", "confused", "curious",
    "desperate", "determined", "disenchanted", "distracted", "drained", "dreamy", "empathetic",
    "enchanted", "energetic", "exhilarated", "focused", "forgiving", "frustrated", "gloomy",
    "grateful", "hateful", "humble", "inspired", "introspective", "jealous", "joyful", "liberated",
    "lonely", "loving", "melancholic", "mischievous", "motivated", "mournful", "mysterious",
    "nostalgic", "optimistic", "passionate", "pensive", "pessimistic", "playful", "powerless",
    "proud", "rebellious", "regretful", "reluctant", "restless", "romantic", "sarcastic",
    "satisfied", "shocked", "skeptical", "submissive", "sympathetic", "tense", "timid", "trapped",
    "uninspired", "vengeful", "vulnerable", "whimsical", "yearning", "zealous"
]

vocal_styles = [
    "featuring clear male vocals",
    "with emotional female vocals",
    "featuring an epic vocal choir",
    "with a catchy vocal melody",
    "featuring a lead singer",
    "with passionate singing"
]

# Folder location in Drive
base_generated_dir = "/content/drive/MyDrive/Plagiarism-Detection-System/data/generated_audio"
stems_base_dir = "/content/drive/MyDrive/Plagiarism-Detection-System/data/separated_segment_smp/mdx_extra_q"
folders = ["musicgen", "audioldm2", "mgeldm"]

for f in folders:
    os.makedirs(os.path.join(base_generated_dir, f), exist_ok=True)

# Testing files
test_files = [
    "/content/drive/MyDrive/Plagiarism-Detection-System/data/segment_smp/audio/pair_9_comp_51s.wav"
]

print(f"Starting generation for {len(test_files)} segments...")

# Creating random text prompt
song_prompts = {}
for audio_input_path in test_files:
    song_name = os.path.splitext(os.path.basename(audio_input_path))[0]
    song_prompts[song_name] = f"{random.choice(moods)} {random.choice(genres)} song, {random.choice(vocal_styles)}, high quality"


# MUSICGEN
if RUN_MUSICGEN:
    print("\nStarting Generation using MusicGen...")
    model_musicgen = MusicGen.get_pretrained('facebook/musicgen-melody')
    model_musicgen.set_generation_params(duration=20)

    for audio_input_path in test_files:
        song_name = os.path.splitext(os.path.basename(audio_input_path))[0]
        current_prompt = song_prompts[song_name]

        print(f"\nSong name: {song_name} [MusicGen]")
        print(f"Prompt: '{current_prompt}'")

        musicgen_out = os.path.join(base_generated_dir, "musicgen", f"{song_name}_musicgen.wav")
        if os.path.exists(musicgen_out):
            print(f"This MusicGen file already exists. Ignoring...")
        else:
            # Generation
            generate_musicgen(current_prompt, audio_input_path, sr=32000, output_path=musicgen_out, model=model_musicgen)

            # Augmentation & Mixing
            vocals_path = os.path.join(stems_base_dir, song_name, "vocals.wav")
            if os.path.exists(vocals_path):
                aug_vocals_path = os.path.join(base_generated_dir, "musicgen", f"{song_name}_temp_aug.wav")
                augment_vocals(vocals_path, aug_vocals_path)
                mix_vocals_and_instrumental(musicgen_out, aug_vocals_path, musicgen_out) # Overwrite the final
                if os.path.exists(aug_vocals_path):
                    os.remove(aug_vocals_path) # Cleaning

    print("\nCleaning up memory...")
    del model_musicgen
    gc.collect()
    torch.cuda.empty_cache()


# AUDIOLDM2
if RUN_AUDIOLDM2:
    print("\nStarting Generation using AudioLDM2...")
    for audio_input_path in test_files:
        song_name = os.path.splitext(os.path.basename(audio_input_path))[0]
        current_prompt = song_prompts[song_name]

        print(f"\nSong name: {song_name} [AudioLDM2]")

        audioldm_out = os.path.join(base_generated_dir, "audioldm2", f"{song_name}_audioldm2.wav")
        if os.path.exists(audioldm_out):
            print(f"This AudioLDM2 file already exists. Ignoring...")
        else:
            # Generation
            generate_audioldm(current_prompt, audio_input_path, sr=16000, output_path=audioldm_out)

            # Augmentation & Mixing
            vocals_path = os.path.join(stems_base_dir, song_name, "vocals.wav")
            if os.path.exists(vocals_path):
                aug_vocals_path = os.path.join(base_generated_dir, "audioldm2", f"{song_name}_temp_aug.wav")
                augment_vocals(vocals_path, aug_vocals_path)
                mix_vocals_and_instrumental(audioldm_out, aug_vocals_path, audioldm_out) # Overwrite the final
                if os.path.exists(aug_vocals_path):
                    os.remove(aug_vocals_path) # Cleaning


# MGE-LDM
if RUN_MGE_LDM:
    print("\nStarting Generation using MGE-LDM...")
    target_stems = ['bass', 'drums', 'other', 'vocals']

    for audio_input_path in test_files:
        song_name = os.path.splitext(os.path.basename(audio_input_path))[0]
        current_prompt = song_prompts[song_name]

        for stem in target_stems:
            print(f"\nSong name: {song_name} | Stem: {stem} [MGE-LDM]")

            mgeldm_out = os.path.join(base_generated_dir, "mgeldm", f"{song_name}_mgeldm_{stem}.wav")
            stem_path = os.path.join(stems_base_dir, song_name, f"{stem}.wav")

            if os.path.exists(mgeldm_out):
                print(f"This MGE-LDM file already exists. Ignoring...")
            else:
                generate_mgeldm(current_prompt, stem_path, output_path=mgeldm_out)

print("\nCompleted Generation!")

Starting generation for 1 segments...

Starting Generation using MGE-LDM...

Song name: pair_9_comp_51s [MGE-LDM]
Generating MGE-LDM audio with prompt: 'zealous c-pop song, featuring clear male vocals, high quality'
Source separation using Demucs on: /content/drive/MyDrive/Plagiarism-Detection-System/data/segment_smp/audio/pair_9_comp_51s.wav
Seperated successfully.
MGE-LDM output saved to: /content/drive/MyDrive/Plagiarism-Detection-System/data/generated_audio/mgeldm/pair_9_comp_51s_mgeldm.wav

Completed Generation!
